# 2.3.1 — Word count distribuito sul full text di CORD-19

Questo notebook **importa** `word_count.py`: non riscrive l'algoritmo e non lancia
sottoprocessi. Tutta la logica sta nel modulo, qui c'è l'esecuzione e la lettura dei
risultati, così esiste una sola sorgente di verità.

L'algoritmo è quello dell'assignment (§2.3.1):

| fase | cosa produce |
|---|---|
| **Map** | per ogni documento *D*, le coppie `(w, cp(w))` — quante volte la parola `w` compare in *D* |
| **Reduce** | per ogni parola `w`, `c(w) = Σ cp(w)` su tutti i documenti |

Struttura dati: **Bag**, come raccomanda il testo («we recommend utilizing the RDD/Bag
data structure»).

Input: `data/silver/paragraphs` — una riga per paragrafo, già sanificato dalla pipeline
di conversione (vedi `DATA_DICTIONARY.md`).

Le scelte di pulizia del testo sono tutte motivate da misure sul corpus: la
giustificazione riga per riga è nel README di questa cartella.

## 1 · Cluster

Dove gira il calcolo lo decide `cluster.txt` alla root del repo (git-ignored), non il
codice: sul Mac parte un `LocalCluster`, sulla VM un `SSHCluster` sui nodi elencati.
Lo stesso notebook gira nei due posti senza modifiche.

In [ ]:
import sys
import time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "Giulia" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "Giulia"))

from cluster import get_client
import word_count as wc

client, cluster = get_client(repo_root=REPO)
client

## 2 · I dati

**Una partizione è un gruppo di file Parquet, e il worker apre i suoi.** Non
`dd.read_parquet(...).to_bag()`, che sarebbe una riga: attraverso quello il numero di
partizioni non è una manopola che imposti, è una proprietà che scopri — e dipende dalla
forma della query. Misurato su questo corpus: leggere tre colonne con un filtro booleano
in mezzo dava **990** partizioni, la lettura piana a due colonne ne dà **1979**, a parità
di 1979 file.

Il benchmark obbligatorio ha il numero di partizioni come **variabile indipendente**:
un numero deciso dall'ottimizzatore lo squalifica. Raggruppando i file si ottiene
esattamente `k`, senza rimescolamenti da pagare — e produzione e benchmark diventano lo
stesso identico codice.

In [ ]:
INPUT = REPO / "data" / "silver" / "paragraphs"

files = wc.paragraph_files(INPUT)                       # i .parquet, in ordine numerico
paragraphs = wc.read_groups(wc.split_evenly(files, len(files)))   # una partizione per file
print(f"{len(files)} file  ->  {paragraphs.npartitions} partizioni")
paragraphs.take(1)

## 3 · Come si comporta la sanitizzazione

Un controllo a occhio prima di lanciare il calcolo vero: cosa resta di un testo con
trattini tipografici, lettere greche e stop-word.

In [ ]:
demo = "The SARS–CoV-2 virus and TNF-α were measured at 5 µg/mL in Müller's study."
print(wc.sanitize(demo))
print(wc.words(demo))

## 4 · Il grafo delle due fasi

`word_count` restituisce i due Bag, ancora **lazy**: nulla è stato calcolato.

**La fase Map non fa passare niente in rete.** `doc_counts` ha lo stesso numero di
partizioni dell'input: il conteggio per documento avviene *dentro* la partizione, ed è il
*combiner* del MapReduce classico. Ciò che esce dal worker è una entry per
`(documento, parola)`, non una per occorrenza.

**La fase Reduce ha due formulazioni**, che calcolano la stessa identica cosa
(verificato sul corpus intero: 6.037.808 parole, 785.753.529 occorrenze, ogni conteggio
uguale) ma si comportano in modo molto diverso:

| | `split_out=0` — `Bag.foldby` | `split_out=16` — groupby DataFrame |
|---|---|---|
| partizioni in uscita | **1** | 16 |
| coda del calcolo | un task solo, seriale | 16 task in parallelo |
| memoria del task finale | tutto il vocabolario (~1,5–2 GB) | ~vocabolario/16 |

`Bag.foldby` e `Bag.frequencies` riducono **sempre** a una partizione sola: tutto lo
spazio delle chiavi deve stare in un singolo task, su un singolo worker. Con la parola
come chiave è fattibile — il vocabolario satura al crescere del corpus (legge di
Heaps) — ma resta una coda seriale. Misurato sul corpus intero:

```
7,0 GB/worker,  foldby         1128 s,  0 worker uccisi
3,4 GB/worker,  foldby         1762 s,  3 worker uccisi
3,4 GB/worker,  split_out=16    274 s,  0 worker uccisi
```

Da qui il default. Il testo dell'assignment permette esplicitamente di passare da Bag a
DataFrame, e la fase Map — dove sta il lavoro vero — resta Bag.

In [ ]:
doc_counts, global_counts = wc.word_count(paragraphs)
print("input        partitions:", paragraphs.npartitions)
print("doc_counts   partitions:", doc_counts.npartitions, "  (Map: nessuno shuffle)")
print("global_counts partitions:", global_counts.npartitions)

_, foldby_counts = wc.word_count(paragraphs, split_out=0)
print("con split_out=0        :", foldby_counts.npartitions, "  (foldby: coda seriale)")

## 5 · Smoke test

Prima del run completo, le stesse identiche operazioni su poche partizioni.

In [ ]:
smoke = wc.read_groups(wc.split_evenly(files[:8], 8))
_, smoke_counts = wc.word_count(smoke)
smoke_counts.topk(10, key=1).compute()

## 6 · Run completo

`topk` pota presto: ogni partizione inoltra solo le sue prime N, quindi non serve
materializzare tutto il vocabolario per avere la classifica.

In [ ]:
TOP_N = 20

started = time.perf_counter()
top = global_counts.topk(TOP_N, key=1).compute()
elapsed = time.perf_counter() - started
print(f"{elapsed:.1f} s")

for word, count in top:
    print(f"{count:>12,}  {word}")

## 7 · Verifica dell'invariante

Il Reduce **raggruppa e basta**: non perde né inventa occorrenze. Il controllo somma i
conteggi prima e dopo e verifica che coincidano.

Nessun totale assoluto scritto a mano: il dump locale e il corpus della VM sono dataset
diversi, quindi l'unica garanzia sensata è strutturale (`PROJECT_CONTEXT.md`, regola 8.2).

Costa: sommare i conteggi per-documento obbliga a percorrere tutta la tabella
intermedia, mentre `topk` può potare. Si lancia in sviluppo e prima di una consegna,
non a ogni esecuzione.

In [ ]:
import dask

after_map, after_reduce = dask.compute(doc_counts.pluck(1).sum(), global_counts.pluck(1).sum())
assert after_map == after_reduce, (after_map, after_reduce)
print(f"invariante ok: {after_map:,} occorrenze prima e dopo il reduce")

## 8 · Barplot

Il grafico che l'assignment chiede esplicitamente («create a barplot of the top
words»).

In [ ]:
OUT = Path("~/mapd-out/word_count").expanduser()
OUT.mkdir(parents=True, exist_ok=True)

wc.barplot(top, OUT / "top_words.png", f"Top {len(top)} words in the CORD-19 body text")

from IPython.display import Image
Image(str(OUT / "top_words.png"))

## 9 · Benchmark obbligatori

Le linee guida del corso li richiedono esplicitamente e senza il progetto è considerato
incompleto: bisogna studiare come le metriche di performance («at least the execution
time») dipendono dai parametri del cluster — «at least the number of dataset partitions
and the number of executors/processing units».

Quel **«processing units»** si legge in due modi — le macchine e i thread — e li
misuriamo tutti e due. In tutto cinque misure:

| | curva | variabile | obbligo |
|---|---|---|---|
| §9.1 | `partizioni` | il numero di partizioni `k` | **sì** |
| §9.2 | `worker` | quante macchine lavorano | **sì** |
| §9.3 | `thread` | quanti thread ha ogni worker | no — è l'altra lettura di «processing units» |
| §9.4 | `foldby` | dove finisce il Reduce | no — riproduce sul cluster vero un confronto che avevamo solo dal Mac |
| §9.5 | — | le fotografie `performance_report` | no |

**Qui non si misura niente: si legge e si disegna.** La divisione è netta e voluta:

| | dove | cosa fa |
|---|---|---|
| misurare | `Giulia/bench_word_count.py`, sul cluster | scrive un CSV, una riga per misura |
| capire | questo notebook | legge il CSV e disegna |

Serve a due cose concrete: la campagna dura ore e non deve dipendere da un notebook
aperto, e per rifare un grafico non si rioccupa il cluster.

```bash
tmux new -s bench
source ~/pyvenv/bin/activate && cd ~/MAPD-Project
python Giulia/bench_word_count.py ~/mapd-data/silver/paragraphs 2>&1 | tee ~/bench.log
```

### Le tre scelte che decidono cosa significano queste curve

**Una riga del CSV = una misura = un cluster nuovo.** Costa un minuto a punto. In cambio
ogni punto parte da worker *appena nati*, e non è pignoleria: su questo cluster un worker
che ha già macinato milioni di stringhe trattiene RSS per frammentazione dell'allocatore
(`PROJECT_CONTEXT.md` §7, Atto 3). Riusando lo stesso cluster per nove punti, l'ultimo
girerebbe su worker stanchi e la misura sommerebbe il partizionamento e l'usura. Come
effetto collaterale sparisce anche il problema di `SSHCluster`, dove `scale()` può solo
scendere e obbligherebbe a ricordarsi un ordine di esecuzione.

**Tutte le misure sono ancorate a un unico punto di riferimento**, ripetuto **3 volte**:
tutti i worker, `k=256`, `split_out=16`, thread di default. Non è un costo extra — è il
punto in comune ai tre assi (è il `k=256` di §9.1, il «tutti i worker» di §9.2, il thread
di default di §9.3). Le 3 ripetizioni servono a misurare **il rumore una volta sola**,
invece di pagarlo su ogni punto: altrove basta una ripetizione, e le barre d'errore di
questo punto dicono quanto vale la differenza fra due punti qualsiasi.

**L'ordine di esecuzione è progettato per una notte che può interrompersi**: prima il
riferimento (che calibra la stima dei tempi), poi la curva a costo prevedibile (worker),
poi quella sulle partizioni **dal centro verso i bordi** — `256 → 512 → 128 → 1024 → 64 →
1979 → 32 → 16 → 8 → 4`. I valori bassi sono i più lenti e i più fragili, quindi stanno in
fondo: se la campagna si ferma, quello che resta in mano è il minimo della curva con i due
rami vicini, cioè la parte che risponde alla domanda.

Il lavoro cronometrato è **quello che si consegna** — Map, Reduce e scrittura del
vocabolario — non una versione più comoda da misurare: il crash dell'11 agosto stava
proprio nella scrittura.

In [ ]:
import pandas as pd

MISURE = Path("~/mapd-out/bench/misure.csv").expanduser()
misure = pd.read_csv(MISURE) if MISURE.exists() else pd.DataFrame()

# Le righe senza `secondi` non sono buchi: sono configurazioni che NON HANNO COMPLETATO,
# ed è un risultato. Restano nel CSV e si contano; spariscono solo dai grafici.
if misure.empty:
    print(f"nessuna misura in {MISURE} — la campagna non è ancora stata lanciata")
else:
    for curva, gruppo in misure.groupby("curva"):
        fallite = gruppo["secondi"].isna().sum()
        print(f"{curva:<12} {len(gruppo):>3} misure, {fallite} non completate, "
              f"{gruppo['valore'].nunique()} punti")

### 9.1 · Tempo vs numero di partizioni *(obbligatorio)*

**Gli stessi identici dati, tagliati in modo diverso.** Cluster fisso, `split_out` fisso:
si muove solo `k`. A dati fissi «numero di partizioni» *è* «dimensione di una partizione».

Il riferimento su cui leggere l'asse x sono gli **slot di calcolo** = worker × thread per
worker. Con 4 worker da 4 thread sono 16: `k=16` è «una partizione per slot», `k=1979` è
«124 per slot».

#### Metà della curva non completa, e non è un buco

Sotto `k=128` i worker muoiono con `KilledWorker`. **Avevo previsto che `k=4` sarebbe
passato**, calcolando ~2,4 GB di testo per partizione contro worker da 6,8 GB. Previsione
sbagliata, e l'errore era misurare la cosa sbagliata: **il testo in ingresso non è ciò che
riempie il worker, lo è l'uscita del Map.**

`Giulia/misura_ram.py` esegue una singola partizione fuori dal cluster e misura la RSS
passo per passo. Sul corpus completo:

| k | testo | coppie Map | **picco 1 task** | × 4 thread | tetto | esito reale |
|---:|---:|---:|---:|---:|---:|---|
| 512 | 0,03 GB | 834 k | 0,50 GB | 2,0 GB | 7,1 | ✓ 490 s |
| 256 | 0,06 GB | 1,81 M | 1,00 GB | 4,0 GB | 7,1 | ✓ 498 s |
| 128 | 0,12 GB | 3,75 M | 1,94 GB | **7,8 GB** | 7,1 | ✓ 509 s *(sul filo)* |
| 64 | 0,29 GB | 7,93 M | 4,07 GB | **16,3 GB** | 7,1 | ✗ `KilledWorker` |

Il muro cade dove la misura lo mette. A `k=128` servirebbero 7,8 GB contro 7,1 e passa lo
stesso, perché Dask mette in pausa i task quando il worker si riempie e i quattro non
arrivano al picco insieme; a `k=64` ne servono 2,3 volte il tetto e non c'è pausa che
tenga.

**Dove va la memoria**, a `k=64`: la fase Map da sola vale il 44% del picco, con
**~230 byte per ogni coppia** `((cord_uid, parola), conteggio)` — misurati identici a
ogni scala, campione compreso. Sono 230 byte per due stringhe corte e un intero: è il
costo dell'**oggetto Python**, non del dato. Il picco complessivo di un task è ~**15×** il
testo che sta elaborando.

Da cui la conseguenza che lega questa sezione alla §9.3: **un thread non è solo un'unità
di calcolo, è un moltiplicatore di memoria.** Quattro thread nello stesso worker vogliono
quattro volte quel picco, e per questo la stessa curva rifatta a 1 thread per worker
sposta il muro di 4× (`--thread 1`). Anche così, `k=32` resterebbe fuori portata: servono
~8 GB per un singolo task, più del tetto di un worker.

> **Il ramo sinistro non è recuperabile su questo hardware, ed è il risultato.** Non è una
> campagna incompleta: è un confine misurato, con un meccanismo e dei numeri. Nemmeno un
> flavor da 16 GB lo sposterebbe di più di un punto.

#### Cosa aspettarsi dal resto

- **a destra la curva risale piano**: i task diventano più corti del tempo di distribuirli,
  e con `k=1979` e `split_out=16` il grafo ha ~35.000 task. Più fine di un file per
  partizione non si può andare (un Parquet del silver ha un solo row-group);
- **il minimo va riportato come *partizioni per slot***, non come numero assoluto: è la
  forma che si trasferisce a un cluster di taglia diversa;
- **le barre d'errore ci sono su un punto solo**, `k=256`, il riferimento ripetuto 3 volte.
  Quella dispersione (±1,4%) è la scala con cui si giudica ogni differenza del grafico.

In [ ]:
import matplotlib.pyplot as plt

# Ogni misura e' un esperimento a UNA variabile. Le curve si costruiscono filtrando sullo
# STATO REALE del cluster (`worker`, `thread`, `partizioni`) e non sull'etichetta `curva`,
# che dice soltanto perche' quella misura e' stata fatta. Due conseguenze utili:
#   - il punto di RIFERIMENTO appartiene a tutte e tre le curve, e ci finisce da solo;
#   - due campagne con thread diversi restano DUE CURVE, invece di essere mediate insieme.
# `thread` nel CSV e' il totale del cluster: i thread per worker (`tpw`) si ricavano.
COLORI = {4: "#2f6f73", 2: "#b4674d", 1: "#7a5c9e"}


def prepara(misure):
    dati = misure.copy()
    dati["tpw"] = (dati["thread"] / dati["worker"]).round().astype(int)
    return dati[dati["split_out"] == 16]        # il foldby e' un'altra domanda (§9.4)


def disegna(ax, frame, x, etichetta, colore):
    """Media delle ripetizioni, con la dispersione min-max come barra d'errore.

    La dispersione non e' decorazione: c'e' su un punto solo - il riferimento, ripetuto
    3 volte - ed e' la scala con cui si giudica ogni differenza del grafico. Due punti
    che distano meno di quella barra non sono diversi."""
    gruppi = frame.dropna(subset=["secondi"]).groupby(x)["secondi"]
    xs = sorted(gruppi.groups)
    medie = gruppi.mean().loc[xs]
    ax.errorbar(xs, medie, marker="o", capsize=4, color=colore, label=etichetta,
                yerr=[medie - gruppi.min().loc[xs], gruppi.max().loc[xs] - medie])
    return medie


def fallite(frame, x):
    """I valori che NON hanno completato. Sono un risultato, non un buco."""
    ko = frame[frame["secondi"].isna()]
    return [int(v) for v in sorted(ko[x].unique())] if not ko.empty else []


if not misure.empty:
    dati = prepara(misure)
    PIENO = int(dati["worker"].max())
    NORMALE = int(dati["tpw"].max())         # i thread di default: i core del nodo
    print(f"cluster pieno: {PIENO} worker · thread di default: {NORMALE} per worker\n")

    partizioni = dati[dati["worker"] == PIENO]
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.set_xscale("log", base=2)
    for tpw in sorted(partizioni["tpw"].unique(), reverse=True):
        ramo = partizioni[partizioni["tpw"] == tpw]
        if ramo.dropna(subset=["secondi"])["partizioni"].nunique() < 2:
            continue                          # un punto solo non e' una curva
        medie = disegna(ax, ramo, "partizioni", f"{tpw} thread per worker",
                        COLORI.get(tpw, "#888"))
        slot = tpw * PIENO
        print(f"{tpw} thread per worker  ({slot} slot di calcolo)")
        print(f"   minimo: {medie.min():.0f} s a {medie.idxmin():.0f} partizioni "
              f"= {medie.idxmin() / slot:.0f} per slot")
        ko = fallite(ramo, "partizioni")
        if ko:
            print(f"   NON hanno completato: k = {ko}")
    ax.set_xlabel("numero di partizioni"); ax.set_ylabel("secondi")
    ax.set_title("Word count: tempo vs partizioni"); ax.grid(alpha=0.25)
    ax.set_ylim(bottom=0); ax.legend()
    plt.show()

    display(partizioni.dropna(subset=["secondi"])
            .groupby(["tpw", "partizioni"])["secondi"]
            .agg(["mean", "min", "max", "count"]).round(1))

### 9.2 · Tempo vs numero di worker *(obbligatorio)*

**Stessi dati, stesso taglio: cambia solo quanta macchina lavora.** I dati sono replicati
sul disco di ogni VM, quindi ogni worker legge sempre da casa propria: togliere worker non
sposta un collo di bottiglia sulla rete, e la curva parla di calcolo. (Con l'architettura
NFS precedente non sarebbe stato vero.)

Le due grandezze che il corso chiede davvero dietro «tempo vs numero di esecutori»:

- **speedup** *S(n) = T(1)/T(n)* — quante volte va più veloce di un worker solo;
- **efficienza** *E(n) = S(n)/n* — quanta parte di ogni worker aggiunto sta davvero
  producendo. Vale 1 se il guadagno è perfetto.

L'efficienza è quella onesta: cala sempre, e **quanto** cala misura la parte non
parallela del lavoro — cioè, qui, la coda del Reduce.

In [ ]:
if not misure.empty:
    # La curva sui worker tiene fissi il partizionamento E i thread per worker: senza il
    # secondo vincolo ci finirebbero dentro anche i punti della misura §9.3, che hanno
    # lo stesso k ma meno thread.
    worker = dati[(dati["partizioni"] == 256) & (dati["tpw"] == NORMALE)]

    fig, (sopra, sotto) = plt.subplots(2, 1, figsize=(7, 7.5), sharex=True)
    medie = disegna(sopra, worker, "worker", f"{NORMALE} thread per worker", COLORI[NORMALE])
    sopra.set_ylabel("secondi"); sopra.set_title("Word count: tempo vs numero di worker")
    sopra.grid(alpha=0.25); sopra.set_ylim(bottom=0)

    base = medie.index.min()                  # il punto piu' piccolo che ha completato
    speedup = medie[base] / medie
    ideale = medie.index / base
    sotto.plot(medie.index, speedup, marker="o", color="#2f6f73", label="speedup misurato")
    sotto.plot(medie.index, ideale, "--", color="#999", label="speedup ideale")
    sotto.plot(medie.index, speedup / ideale, marker="s", color="#b4674d", label="efficienza")
    sotto.set_xlabel("numero di worker"); sotto.grid(alpha=0.25)
    sotto.set_ylim(bottom=0); sotto.legend()
    plt.show()

    display(pd.DataFrame({"secondi": medie.round(1),
                          "speedup": speedup.round(2),
                          "efficienza": (speedup / ideale).round(2)}))

    # La frazione seriale secondo Amdahl, ricavata da ogni punto: S(n) = 1/(s+(1-s)/n).
    # Se i valori non coincidono, non e' solo lavoro seriale - c'e' anche un costo di
    # coordinamento che CRESCE col numero di worker.
    seriale = ((medie.index / speedup - 1) / (medie.index - 1)).round(3)
    print("frazione seriale stimata da ogni punto:",
          {int(n): s for n, s in zip(medie.index, seriale) if n > 1})

### 9.3 · Tempo vs thread per worker — l'altra lettura di «processing units»

Il testo del corso chiede il numero di «executors/**processing units**». §9.2 risponde
contando le **macchine**; qui contiamo le **unità di calcolo dentro una macchina**. Sono
due domande diverse, e su questo carico danno risposte diverse — che è precisamente il
motivo per cui vale la pena misurarle entrambe.

**L'ipotesi, scritta prima di guardare il grafico.** La fase Map è `re.findall` +
`Counter` in Python puro, e in CPython quel codice **tiene il GIL**: due thread dello
stesso processo non tokenizzano davvero in parallelo, si alternano. L'unica parte che il
GIL lo rilascia è la lettura Parquet, che è dentro `pyarrow`. Quindi:

> **Previsione:** `T(4 thread) / T(1 thread) ≈ 0,6`, non `0,25`. Cioè quadruplicare i
> thread NON quadruplica la velocità. La curva sui worker (§9.2) invece scala molto
> meglio, perché ogni worker è un **processo separato**, con il suo GIL.

Le due letture possibili del risultato, decise in anticipo:

- **se la previsione regge**: su questo carico l'unità di calcolo utile è il **processo**,
  non il thread. Una macchina con 4 core e un worker solo ne usa ~1,5. La conseguenza
  pratica — un worker per core invece di un worker con 4 thread — è misurabile ripetendo
  l'IP in `cluster.txt`, ed è la direzione naturale del seguito;
- **se viene smentita** (cioè se scala bene): il tempo non è dominato dalla tokenizzazione
  ma da I/O e decompressione Arrow, che il GIL lo rilasciano. Sarebbe un risultato
  altrettanto informativo, e cambierebbe dove cercare il collo di bottiglia.

Attenzione all'asse x: nel CSV la colonna `thread` è il **totale del cluster**, quindi qui
si divide per il numero di worker per avere i thread di *ogni* worker.

In [ ]:
if not misure.empty:
    # Cluster pieno e partizionamento fisso: si muove solo quanti thread ha ogni worker.
    thread = dati[(dati["partizioni"] == 256) & (dati["worker"] == PIENO)]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    medie = disegna(ax, thread, "tpw", None, "#2f6f73")
    ax.set_xlabel("thread per worker"); ax.set_ylabel("secondi")
    ax.set_title("Word count: tempo vs thread per worker")
    ax.grid(alpha=0.25); ax.set_ylim(bottom=0)
    plt.show()

    uno, tanti = medie.index.min(), medie.index.max()
    display(pd.DataFrame({
        "secondi": medie.round(1),
        "core del cluster": medie.index * PIENO,
        "guadagno reale": (medie[uno] / medie).round(2),
        "guadagno se scalasse": (medie.index / uno).round(2),
    }))
    print(f"T({tanti} thread) / T({uno} thread) = {medie[tanti] / medie[uno]:.2f}")
    print(f"   previsto ~0,60 se il Map tiene il GIL · ~{uno / tanti:.2f} se scalasse")
    print(f"   sopra 1,00 vuol dire che i thread RALLENTANO: non sono unita' di calcolo,")
    print(f"   sono moltiplicatori di memoria (§9.1 e Giulia/misura_ram.py)")

### 9.4 · Dove finisce il Reduce, sul cluster vero

Non è una curva: sono **due punti**, identici in tutto tranne `split_out`. È la stessa
configurazione del riferimento, con la riduzione finale spostata dal `groupby` a 16 shard
al `Bag.foldby` che riduce a **una partizione sola**.

Il confronto lo avevamo già, ma **misurato sul Mac** (§10): 1.128 s contro 274 s, cioè
4,1×. Rifarlo qui costa una riga della campagna e vale molto di più, perché sul cluster
vero c'è la rete di mezzo: quando un solo task macina un dizionario da 6 milioni di voci,
gli altri worker non sono solo fermi — sono fermi *e* devono spedirgli tutto.

**Questa misura è l'ultima della campagna, ed è sacrificabile per costruzione.** Il
`foldby` sul corpus intero è già morto una volta con `KilledWorker`. Se non completa, la
riga resta nel CSV con il suo errore, ed è il risultato più forte possibile: la stessa
identica funzione matematica, che nell'unica formulazione «naturale» in Bag *non arriva in
fondo*.

In [ ]:
if not misure.empty:
    reduce = misure[misure["curva"].isin(["riferimento", "foldby"])]
    display(reduce[["curva", "split_out", "secondi", "errore", "partizioni", "worker", "thread"]])

    a_shard = reduce.loc[reduce["curva"] == "riferimento", "secondi"].mean()
    a_foldby = reduce.loc[reduce["curva"] == "foldby", "secondi"].mean()
    if pd.notna(a_foldby) and pd.notna(a_shard):
        print(f"split_out=16: {a_shard:.0f} s   foldby: {a_foldby:.0f} s   "
              f"-> {a_foldby / a_shard:.1f}x piu' lento, a parita' di risultato")
    elif pd.notna(a_shard):
        print(f"split_out=16: {a_shard:.0f} s   foldby: NON HA COMPLETATO")

### 9.5 · Le fotografie: una per ogni misura

Oltre ai numeri, ogni misura della campagna produce una pagina HTML con
`performance_report`. Costa due righe e dà quello che nessuna curva può dare: la **linea
del tempo di ogni task su ogni worker** (scheda *Task Stream*) e la **memoria di ogni
worker nel tempo** (scheda *System*).

**Ce n'è una per punto, non quattro scelte in anticipo.** È il motivo per cui il picco di
memoria non sta nel CSV: quel dato è già nel report di ogni singola misura, e non serve
decidere prima quale servirà. Il nome dice qual è:

```
report_<curva>_<valore>_<ripetizione>.html
```

I confronti da guardare, e cosa ci si vede:

| accosta | e | mostra |
|---|---|---|
| `report_partizioni_4_0` | `report_riferimento_4_0` | a `k=4` il Task Stream ha **quattro barre e il resto bianco**: i thread senza lavoro |
| `report_partizioni_1979_0` | `report_riferimento_4_0` | a `k=1979` le barre diventano un pettine fittissimo: il costo dello scheduling |
| `report_foldby_0_0` | `report_riferimento_4_0` | **la coda seriale**: un task solo che macina, tutti gli altri worker fermi |
| `report_worker_1_0` | `report_worker_3_0` | il cluster mezzo vuoto contro il cluster pieno |
| `report_thread_1_0` | `report_riferimento_4_0` | quanto lavoro *vero* fa un worker a 4 thread rispetto a uno a 1 thread |

I file si aprono nel browser. Sono autoconsistenti (`mode="inline"`): senza quel parametro
l'HTML scarica BokehJS da `cdn.bokeh.org` e resta **bianco** appena lo si guarda senza
internet — cioè, tipicamente, dopo averlo copiato giù dal cluster, che è l'unico momento
in cui lo si guarda. In tutto sono ~75 MB: si scaricano insieme al CSV.

## 10 · Dove finisce il Reduce

Non è fra i benchmark obbligatori ed è il risultato più istruttivo del task. Non ha avuto
bisogno di nessuna impalcatura: sono **due lanci di `word_count.py` con un flag diverso**,
sul corpus intero, e i numeri scritti a mano.

| configurazione | tempo | worker uccisi | risultato |
|---|---:|---:|---|
| 7,0 GB per worker, `foldby` | 1.128 s | 0 | 6.037.808 parole |
| 3,4 GB per worker, `foldby` | 1.762 s | **3** | identico |
| 3,4 GB per worker, `split_out=16` | **274 s** | 0 | identico |

**4,1× più veloce del miglior run con `foldby`, su worker grandi la metà.** Stesso
risultato, chiave per chiave.

La lettura giusta non è «servivano più GB»: `Bag.foldby` restituisce **una** partizione
per costruzione — è l'ultimo argomento nel sorgente di dask, `dask/bag/core.py:1418` —
quindi un solo task macina un dizionario Python da 6 milioni di voci mentre gli altri
worker stanno fermi. Con `split_out` diventano 16 task in parallelo, ognuno con circa un
sedicesimo del vocabolario, in colonne Arrow invece che in oggetti Python. La memoria era
il sintomo; il collo di bottiglia era il parallelismo.

**Morale che vale per tutti e quattro i task:** un oggetto grosso dev'essere il *risultato
di tanti task piccoli*, mai la *variabile locale di un task grosso* — perché solo il primo
Dask lo sa gestire, spostare e riversare su disco.

## 11 · Chiusura

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("cluster chiuso")